# Invoice Data Cleaning

## Objective

Clean and validate invoice transaction data before loading into the analytics database.

## Cleaning Tasks

- Identify missing values
- Validate invoice IDs
- Detect duplicate invoices
- Validate dates
- Check for negative quantities
- Recalculate revenue
- Verify transaction integrity
- Export cleaned invoice data

In [6]:
import pandas as pd
from pathlib import Path

In [7]:
raw_path = Path("../data/raw/invoices.csv")

clean_path = Path("../data/cleaned/invoices_clean.csv")

In [8]:
invoices = pd.read_csv(raw_path)

invoices.head()

,invoice_id,invoice_date,customer_id,product_id,warehouse,sales_rep,quantity_cases,case_price,discount_pct,revenue
0,500001,2025-12-23,497.0,1005,Los Angeles DC,Morgan Lee,34,36.69,0.00,1247.46
1,500002,2025-03-10,163.0,1015,Phoenix DC,Ashley Nguyen,162,35.03,0.05,5391.12
2,500003,2025-11-10,104.0,1014,Phoenix DC,Ryan Thompson,786,33.87,0.05,25290.73
3,500004,2025-08-17,201.0,1019,Atlanta DC,Casey Davis,73,39.05,0.05,2708.12
4,500005,2025-11-23,46.0,1008,Phoenix DC,Casey Davis,36,36.32,0.00,1307.52


In [9]:
invoices.shape

(50000, 10)

In [10]:
invoices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      50000 non-null  int64  
 1   invoice_date    50000 non-null  object 
 2   customer_id     49998 non-null  float64
 3   product_id      50000 non-null  int64  
 4   warehouse       50000 non-null  object 
 5   sales_rep       50000 non-null  object 
 6   quantity_cases  50000 non-null  int64  
 7   case_price      50000 non-null  float64
 8   discount_pct    50000 non-null  float64
 9   revenue         50000 non-null  float64
dtypes: float64(4), int64(3), object(3)
memory usage: 3.8+ MB


In [11]:
invoices.isnull().sum()

invoice_id        0
invoice_date      0
customer_id       2
product_id        0
warehouse         0
sales_rep         0
quantity_cases    0
case_price        0
discount_pct      0
revenue           0
dtype: int64

In [12]:
invoices["invoice_id"].duplicated().sum()

1

In [13]:
invoices[
    invoices["quantity_cases"] <= 0
]

,invoice_id,invoice_date,customer_id,product_id,warehouse,sales_rep,quantity_cases,case_price,discount_pct,revenue
1000,501001,2025-06-07,448.0,1005,Los Angeles DC,Jordan Kim,-250,36.69,0.1,10797.87
2000,502001,2025-11-03,286.0,1013,Los Angeles DC,Casey Davis,-50,34.90,0.0,4746.40


In [14]:
invoices["invoice_date"].min()

'2025-01-01'

In [15]:
invoices["invoice_date"].max()

'2025-15-99'

In [16]:
invoices[
    invoices["invoice_id"].duplicated(keep=False)
]

,invoice_id,invoice_date,customer_id,product_id,warehouse,sales_rep,quantity_cases,case_price,discount_pct,revenue
99,500100,2025-07-22,199.0,1028,Phoenix DC,Alex Martinez,311,34.61,0.00,10763.71
100,500100,2025-05-27,67.0,1016,Dallas DC,Ashley Nguyen,410,34.52,0.05,13445.54


In [17]:
invoices_clean = invoices.copy()

In [18]:
products_clean = pd.read_csv(
    "../data/cleaned/products_clean.csv"
)

In [19]:
invoices_clean["invoice_date"] = pd.to_datetime(
    invoices_clean["invoice_date"],
    errors="coerce"
)

In [20]:
invoices_clean["invoice_date"].isnull().sum()

1

In [21]:
invoices[
    invoices["invoice_id"].duplicated(keep=False)
]

,invoice_id,invoice_date,customer_id,product_id,warehouse,sales_rep,quantity_cases,case_price,discount_pct,revenue
99,500100,2025-07-22,199.0,1028,Phoenix DC,Alex Martinez,311,34.61,0.00,10763.71
100,500100,2025-05-27,67.0,1016,Dallas DC,Ashley Nguyen,410,34.52,0.05,13445.54


In [22]:
invoices_clean[
    invoices_clean["invoice_date"].isna()
]

,invoice_id,invoice_date,customer_id,product_id,warehouse,sales_rep,quantity_cases,case_price,discount_pct,revenue
4000,504001,NaT,454.0,1014,Phoenix DC,Alex Martinez,58,33.87,0.05,1866.24


In [23]:
invoices_clean.loc[
    invoices_clean["invoice_id"].duplicated(),
    "invoice_id"
] = invoices_clean["invoice_id"].max() + 1

In [24]:
invoices_clean["invoice_id"].duplicated().sum()

0

In [25]:
invoices_clean.loc[
    invoices_clean["invoice_id"].duplicated(),
    "invoice_id"
] = invoices_clean["invoice_id"].max() + 1

In [26]:
invoices_clean["invoice_id"].duplicated().sum()

0

In [27]:
invoices_clean[
    invoices_clean["quantity_cases"] <= 0
]

,invoice_id,invoice_date,customer_id,product_id,warehouse,sales_rep,quantity_cases,case_price,discount_pct,revenue
1000,501001,2025-06-07,448.0,1005,Los Angeles DC,Jordan Kim,-250,36.69,0.1,10797.87
2000,502001,2025-11-03,286.0,1013,Los Angeles DC,Casey Davis,-50,34.90,0.0,4746.40


In [28]:
invoices_clean["quantity_cases"] = (
    invoices_clean["quantity_cases"]
    .abs()
)

In [29]:
# Replace invoice prices with validated product prices

invoices_clean = invoices_clean.drop(
    columns=["case_price"]
)

invoices_clean = invoices_clean.merge(
    products_clean[["product_id", "case_price"]],
    on="product_id",
    how="left"
)

In [30]:
invoices_clean[
    invoices_clean["quantity_cases"] <= 0
]

,invoice_id,invoice_date,customer_id,product_id,warehouse,sales_rep,quantity_cases,discount_pct,revenue,case_price


Invoice Data Quality Findings:

- Identified 2 negative quantity transactions.
- Corrected invalid quantity signs while preserving transaction records.

In [31]:
invoices_clean["expected_revenue"] = (
    invoices_clean["quantity_cases"]
    * invoices_clean["case_price"]
    * (1 - invoices_clean["discount_pct"])
)

In [32]:
invoices_clean["revenue_difference"] = (
    invoices_clean["revenue"]
    - invoices_clean["expected_revenue"]
)

In [33]:
(invoices_clean["revenue_difference"] != 0).sum()

20440

In [34]:
invoices_clean[
    invoices_clean["revenue_difference"] != 0
].head()

,invoice_id,invoice_date,customer_id,product_id,warehouse,sales_rep,quantity_cases,discount_pct,revenue,case_price,expected_revenue,revenue_difference
1,500002,2025-03-10,163.0,1015,Phoenix DC,Ashley Nguyen,162,0.05,5391.12,35.03,5391.1170,0.0030
2,500003,2025-11-10,104.0,1014,Phoenix DC,Ryan Thompson,786,0.05,25290.73,33.87,25290.7290,0.0010
3,500004,2025-08-17,201.0,1019,Atlanta DC,Casey Davis,73,0.05,2708.12,39.05,2708.1175,0.0025
6,500007,2025-06-13,228.0,1011,Los Angeles DC,Taylor Johnson,239,0.05,8285.05,36.49,8285.0545,-0.0045
9,500010,2025-07-19,236.0,1014,Phoenix DC,Alex Martinez,205,0.10,6249.01,33.87,6249.0150,-0.0050


In [35]:
invoices_clean["revenue_difference_abs"] = (
    invoices_clean["revenue_difference"]
    .abs()
)

In [36]:
(invoices_clean["revenue_difference_abs"] > 0.01).sum()

1797

In [37]:
invoices_clean[
    invoices_clean["revenue_difference_abs"] > 0.01
].head()

,invoice_id,invoice_date,customer_id,product_id,warehouse,sales_rep,quantity_cases,discount_pct,revenue,case_price,expected_revenue,revenue_difference,revenue_difference_abs
10,500011,2025-07-14,375.0,1020,Chicago DC,Chris Ramirez,738,0.00,29091.96,39.76,29342.880,-250.920,250.920
72,500073,2025-11-08,133.0,1026,Dallas DC,Chris Ramirez,114,0.05,-3344.30,36.41,3943.203,-7287.503,7287.503
88,500089,2025-12-17,347.0,1026,Dallas DC,Jordan Kim,35,0.10,-972.72,36.41,1146.915,-2119.635,2119.635
101,500102,2025-04-08,304.0,1026,Dallas DC,Jordan Kim,212,0.00,-6546.56,36.41,7718.920,-14265.480,14265.480
158,500159,2025-08-08,216.0,1020,Atlanta DC,Chris Ramirez,260,0.00,10249.20,39.76,10337.600,-88.400,88.400


In [38]:
invoices_clean.loc[
    invoices_clean["revenue_difference_abs"] > 0.01,
    "revenue"
] = invoices_clean["expected_revenue"]

In [39]:
invoices_clean["revenue_difference"] = (
    invoices_clean["revenue"]
    - invoices_clean["expected_revenue"]
)

In [40]:
(invoices_clean["revenue_difference"].abs() > 0.01).sum()

0

In [41]:
invoices_clean = invoices_clean.drop(
    columns=[
        "expected_revenue",
        "revenue_difference",
        "revenue_difference_abs"
    ]
)

In [42]:
invoices_clean.isnull().sum()

invoice_id        0
invoice_date      1
customer_id       2
product_id        0
warehouse         0
sales_rep         0
quantity_cases    0
discount_pct      0
revenue           0
case_price        0
dtype: int64

In [43]:
invoices_clean["invoice_id"].duplicated().sum()

0

In [44]:
(invoices_clean["quantity_cases"] <= 0).sum()

0

In [45]:
invoices_clean["revenue_check"] = (
    invoices_clean["quantity_cases"]
    * invoices_clean["case_price"]
    * (1 - invoices_clean["discount_pct"])
)

In [46]:
(
    abs(
        invoices_clean["revenue"] 
        - invoices_clean["revenue_check"]
    ) > 0.01
).sum()

0

In [47]:
invoices_clean = invoices_clean.drop(
    columns=["revenue_check"]
)

In [48]:
invoices_clean.to_csv(
    clean_path,
    index=False
)

In [49]:
invoices_clean["invoice_date"] = (
    invoices_clean["invoice_date"]
    .fillna(pd.Timestamp("2025-01-01"))
)

In [50]:
invoices_clean["customer_id"] = (
    invoices_clean["customer_id"]
    .fillna(0)
    .astype(int)
)

In [51]:
invoices_clean.isnull().sum()

invoice_id        0
invoice_date      0
customer_id       0
product_id        0
warehouse         0
sales_rep         0
quantity_cases    0
discount_pct      0
revenue           0
case_price        0
dtype: int64

In [52]:
invoices_clean.to_csv(
    clean_path,
    index=False
)

In [53]:
invoices_clean.head()

,invoice_id,invoice_date,customer_id,product_id,warehouse,sales_rep,quantity_cases,discount_pct,revenue,case_price
0,500001,2025-12-23,497,1005,Los Angeles DC,Morgan Lee,34,0.00,1247.46,36.69
1,500002,2025-03-10,163,1015,Phoenix DC,Ashley Nguyen,162,0.05,5391.12,35.03
2,500003,2025-11-10,104,1014,Phoenix DC,Ryan Thompson,786,0.05,25290.73,33.87
3,500004,2025-08-17,201,1019,Atlanta DC,Casey Davis,73,0.05,2708.12,39.05
4,500005,2025-11-23,46,1008,Phoenix DC,Casey Davis,36,0.00,1307.52,36.32


In [54]:
invoices_clean.shape

(50000, 10)

## Invoice Cleaning Summary

The invoice transaction dataset was cleaned and validated through:

- Identifying and resolving duplicate invoice identifiers
- Handling missing customer references
- Correcting invalid invoice dates
- Fixing negative quantity transactions
- Validating revenue calculations against quantity, price, and discount logic
- Correcting inaccurate revenue values
- Confirming final data completeness and integrity

The cleaned invoice dataset is ready for analytics and database loading.

In [55]:
import pandas as pd

invoices_clean = pd.read_csv(
    "../data/cleaned/invoices_clean.csv"
)

invoices_clean[
    invoices_clean["product_id"] == 1021
]

,invoice_id,invoice_date,customer_id,product_id,warehouse,sales_rep,quantity_cases,discount_pct,revenue,case_price


In [56]:
import pandas as pd

sales = pd.read_csv(
    "../data/cleaned/invoices_clean.csv"
)

products = pd.read_csv(
    "../data/cleaned/products_clean.csv"
)

missing_products = sales[
    ~sales["product_id"].isin(products["product_id"])
]

missing_products["product_id"].unique()

array([], dtype=int64)

In [57]:
missing_customers = sales[
    ~sales["customer_id"].isin(
        pd.read_csv("../data/cleaned/customers_clean.csv")["customer_id"]
    )
]

missing_customers["customer_id"].unique()

array([0])

In [59]:
invoices_clean[
    invoices_clean["product_id"] == 1026
][["product_id", "case_price"]]

,product_id,case_price
72,1026,36.41
88,1026,36.41
101,1026,36.41
163,1026,36.41
169,1026,36.41
...,...,...
49618,1026,36.41
49630,1026,36.41
49974,1026,36.41
49984,1026,36.41


In [60]:
invoices_clean["revenue"].max()

31768.24

In [61]:
invoices_clean.loc[
    invoices_clean["revenue"].idxmax()
]

invoice_id            522305
invoice_date      2025-08-25
customer_id              222
product_id              1020
warehouse         Atlanta DC
sales_rep         Morgan Lee
quantity_cases           799
discount_pct             0.0
revenue             31768.24
case_price             39.76
Name: 22304, dtype: object

In [62]:
invoices_clean.dtypes

invoice_id          int64
invoice_date       object
customer_id         int64
product_id          int64
warehouse          object
sales_rep          object
quantity_cases      int64
discount_pct      float64
revenue           float64
case_price        float64
dtype: object

In [63]:
invoices_clean.describe()

,invoice_id,customer_id,product_id,quantity_cases,discount_pct,revenue,case_price
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,525001.498000,250.658360,1012.115660,308.791500,0.018512,10839.142892,35.762159
std,14433.904524,144.122268,8.273534,214.538607,0.035824,7587.542728,2.098186
min,500001.000000,0.000000,1001.000000,20.000000,0.000000,573.480000,31.220000
25%,512501.750000,126.000000,1005.000000,79.000000,0.000000,2917.470000,34.430000
50%,525001.500000,250.000000,1011.000000,311.000000,0.000000,10820.685000,35.500000
75%,537501.250000,375.000000,1017.000000,456.000000,0.000000,15987.187500,36.930000
max,550001.000000,500.000000,1034.000000,800.000000,0.150000,31768.240000,39.760000


In [64]:
for col in invoices_clean.columns:
    print(col, invoices_clean[col].max())

invoice_id 550001
invoice_date 2025-12-31
customer_id 500
product_id 1034
warehouse Phoenix DC
sales_rep Taylor Johnson
quantity_cases 800
discount_pct 0.15
revenue 31768.24
case_price 39.76
